# Qwen3-8B 회의록 요약 LoRA 학습
ModuStudy/Squiz - SSAFY L40S 46GB GPU

**D106팀 - GPU Device 3**

## 1. GPU 설정 및 환경 확인

In [ ]:
# SSAFY GPU 서버 설정 (D106팀 - Device 3)
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

# GPU 메모리 정리 (중요!)
import gc
import torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

# GPU 확인
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
print(f"현재 GPU: {torch.cuda.get_device_name(0)}")

total_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
used_mem = torch.cuda.memory_allocated(0) / 1024**3
free_mem = total_mem - used_mem

print(f"GPU 총 메모리: {total_mem:.1f} GB")
print(f"GPU 사용 중: {used_mem:.1f} GB")
print(f"GPU 여유: {free_mem:.1f} GB")

if free_mem < 10:
    print("\n⚠️ GPU 메모리 부족! 커널을 재시작하세요.")

## 2. 설정값 정의

In [ ]:
# ===== 설정 =====
MODEL_NAME = "Qwen/Qwen3-8B"
MAX_SEQ_LENGTH = 4096

# LoRA 설정
LORA_R = 64
LORA_ALPHA = 128
LORA_DROPOUT = 0.05

# 학습 하이퍼파라미터
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 2  # effective batch size = 8
LEARNING_RATE = 2e-4
NUM_EPOCHS = 4
WARMUP_RATIO = 0.1

# 데이터 경로 (루트에 있으면 폴더 경로 제거)
DATA_PATH = "training_data_train.json"
VAL_PATH = "training_data_val.json"
OUTPUT_DIR = "./outputs/qwen3-summarizer"

# 4-bit 양자화 사용
USE_4BIT = True

# 테스트용 입력 (Before/After 비교용)
TEST_INPUT = """다음 IT 스터디 회의 내용을 요약해주세요.

회의 내용:
김민수: 어 네 그럼 오늘 스터디 시작할게요. 오늘은 BFS DFS 문제 풀이 리뷰하기로 했죠?
이지은: 네네 맞아요. 저 백준 1260번 풀어왔는데요 음 DFS랑 BFS 둘 다 구현했거든요.
박준혁: 아 저도요. 근데 그 BFS에서 visited 체크 위치가 좀 헷갈리더라고요.
김민수: 아아 그거 중요한 포인트예요. 큐에 넣을 때 체크해야 중복 방문 안 해요.
이지은: 아 그렇구나. 저는 꺼낼 때 체크했거든요. 그래서 메모리 초과 났었어요.
박준혁: 오 그래서 그랬구나. 저도 같은 실수 했었는데.
김민수: 네 다음 주는 DP 문제 풀어오기로 해요. 백준 1149번 RGB거리요.
"""

print("설정 완료!")
print(f"모델: {MODEL_NAME}")
print(f"LoRA rank: {LORA_R}, alpha: {LORA_ALPHA}")
print(f"Batch size: {BATCH_SIZE} x {GRADIENT_ACCUMULATION_STEPS} = {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"Epochs: {NUM_EPOCHS}")

## 3. 라이브러리 임포트

In [ ]:
import gc
import json
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print("라이브러리 임포트 완료!")

## 4. 데이터 로드

In [ ]:
# 학습 데이터 로드
with open(DATA_PATH, "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(VAL_PATH, "r", encoding="utf-8") as f:
    val_data = json.load(f)

print(f"학습 데이터: {len(train_data)}개")
print(f"검증 데이터: {len(val_data)}개")

# 타입별 분포 확인
from collections import Counter
type_counts = Counter([d['type'] for d in train_data])
print(f"\n타입별 분포:")
for t, c in type_counts.items():
    print(f"  - {t}: {c}개")

# 데이터 샘플 확인
print("\n=== 샘플 데이터 ===")
print(f"Type: {train_data[0]['type']}")
print(f"Text 길이: {len(train_data[0]['text'])} 문자")

## 5. 모델 및 토크나이저 로드

In [ ]:
# GPU 메모리 정리
gc.collect()
torch.cuda.empty_cache()

print(f"[MODEL] {MODEL_NAME} 로딩 중... (몇 분 소요)")

# 토크나이저
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    padding_side="right",
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("토크나이저 로드 완료!")

# 4-bit 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# GPU 메모리 제한 설정 (L40S 46GB 중 40GB 사용)
max_memory = {0: "40GiB", "cpu": "30GiB"}

# 모델 로드
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    max_memory=max_memory,
    trust_remote_code=True,
)

print("모델 로드 완료!")
print(f"모델 메모리: {model.get_memory_footprint() / 1024**3:.2f} GB")
print(f"GPU 사용량: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")

## 6. LoRA 설정

In [ ]:
# LoRA 준비
model = prepare_model_for_kbit_training(model)

# LoRA 설정
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)

# 학습 가능한 파라미터 확인
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"학습 가능 파라미터: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
print(f"전체 파라미터: {total_params:,}")

## 7. ⭐ Before 테스트 (파인튜닝 전)

In [ ]:
def run_inference(model, tokenizer, test_input):
    """추론 실행 함수"""
    prompt = f"""<|im_start|>system
당신은 IT 스터디 회의록을 요약하는 전문가입니다. 핵심 내용을 정확하게 정리합니다.<|im_end|>
<|im_start|>user
{test_input}<|im_end|>
<|im_start|>assistant
"""
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=False)
    assistant_start = response.find("<|im_start|>assistant\n") + len("<|im_start|>assistant\n")
    assistant_end = response.find("<|im_end|>", assistant_start)
    summary = response[assistant_start:assistant_end] if assistant_end != -1 else response[assistant_start:]
    
    return summary.strip()


print("=" * 60)
print("🔵 BEFORE 테스트 (파인튜닝 전)")
print("=" * 60)
print("\n[입력]")
print(TEST_INPUT[:200] + "...")
print("\n[출력]")

before_result = run_inference(model, tokenizer, TEST_INPUT)
print(before_result)
print("\n" + "=" * 60)

## 8. 데이터셋 준비

In [ ]:
def tokenize_function(examples):
    """토크나이징 + labels 마스킹 (assistant 응답만 학습)"""
    result = tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
    )

    # Labels 생성 (assistant 응답 부분만 학습)
    labels = result["input_ids"].copy()

    # assistant 마커 이전 부분 마스킹 (-100)
    text = examples["text"]
    assistant_marker = "<|im_start|>assistant\n"
    idx = text.find(assistant_marker)

    if idx != -1:
        prefix_text = text[:idx + len(assistant_marker)]
        prefix_tokens = tokenizer(prefix_text, add_special_tokens=False)["input_ids"]
        mask_len = min(len(prefix_tokens), len(labels))
        for i in range(mask_len):
            labels[i] = -100

    result["labels"] = labels
    return result


# Dataset 생성
train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

print(f"Train dataset: {len(train_dataset)}개")
print(f"Val dataset: {len(val_dataset)}개")

# 토크나이징
print("\n토크나이징 중...")
train_dataset = train_dataset.map(tokenize_function, remove_columns=train_dataset.column_names)
val_dataset = val_dataset.map(tokenize_function, remove_columns=val_dataset.column_names)

print("토크나이징 완료!")

## 9. 학습 설정

In [ ]:
# 학습 설정
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    report_to="none",
    dataloader_pin_memory=False,
    remove_unused_columns=False,  # 중요: labels 유지
)

# Data Collator - labels 패딩 지원
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    pad_to_multiple_of=8,
    label_pad_token_id=-100,  # labels 패딩은 -100으로
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

print("Trainer 설정 완료!")
print(f"출력 디렉토리: {OUTPUT_DIR}")

## 10. 🚀 학습 시작!

In [ ]:
print("=" * 60)
print("🚀 학습 시작!")
print("=" * 60)
print(f"모델: {MODEL_NAME}")
print(f"데이터: {len(train_dataset)}개 (train) / {len(val_dataset)}개 (val)")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Batch size: {BATCH_SIZE} x {GRADIENT_ACCUMULATION_STEPS} = {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print("=" * 60)

# 학습!
trainer.train()

## 11. 모델 저장

In [ ]:
# 최종 모델 저장
final_output_dir = f"{OUTPUT_DIR}/final"
trainer.save_model(final_output_dir)
tokenizer.save_pretrained(final_output_dir)

print(f"\n모델 저장 완료: {final_output_dir}")
print("\n학습 완료! 🎉")

## 12. ⭐ After 테스트 (파인튜닝 후)

In [ ]:
print("=" * 60)
print("🟢 AFTER 테스트 (파인튜닝 후)")
print("=" * 60)
print("\n[입력]")
print(TEST_INPUT[:200] + "...")
print("\n[출력]")

after_result = run_inference(model, tokenizer, TEST_INPUT)
print(after_result)
print("\n" + "=" * 60)

## 13. 📊 Before vs After 비교

In [ ]:
print("=" * 60)
print("📊 Before vs After 비교")
print("=" * 60)

print("\n🔵 [BEFORE - 파인튜닝 전]")
print("-" * 40)
print(before_result)

print("\n🟢 [AFTER - 파인튜닝 후]")
print("-" * 40)
print(after_result)

print("\n" + "=" * 60)
print("✅ 학습 및 테스트 완료!")
print("=" * 60)